# 02. Vertex AI で動かす Hello World とバックエンドの使い分け

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/ja/02_vertex_ai_quickstart.ipynb)

統合 `google-genai` SDK の最大の強みは、Google AI Studio と Google Cloud Vertex AI の違いが **`genai.Client(...)` の初期化コードのみに局所化されている点**にあります。一度クライアントを生成すれば、`client.models.generate_content`、ストリーミング、`ThinkingConfig`、Pydantic による構造化出力、マルチターンチャット（`client.chats`）、Function Calling まで、後続のすべての呼び出しコードは **100% 同一**のまま動作します。

![図2-1: 単一の google-genai SDK による AI Studio / Vertex AI 統合アーキテクチャ](https://examples.drawlib.com/google-genai/images/02_vertex_ai_quickstart/ch02_unified_sdk_arch_ja.png)

*図2-1: 単一の google-genai SDK による AI Studio / Vertex AI 統合アーキテクチャ*

---

## 1. Vertex AI での最小構成 Hello World（Colab キーレス IAM 認証）

Google Cloud Vertex AI を利用する場合、静的な API キーを発行・配布する必要はありません。ノートブック冒頭の `!pip install -q google-genai` セットアップセルを実行したあと、Google Colab 上で `auth.authenticate_user()` を呼び出すと、ブラウザの認証ポップアップを通じて Google Cloud IAM に紐づく一時的な認証情報（ADC: Application Default Credentials）が自動的に設定されます。事前に対象プロジェクトで Vertex AI API（`aiplatform.googleapis.com`）を有効化し、使用する Google アカウントに **Vertex AI ユーザー**（`roles/aiplatform.user`）ロールを付与したうえで、セル先頭の `GC_PROJECT` をご自身のプロジェクト ID に書き換えて実行してください（`LOCATION = "global"` と `MODEL = "gemini-3.8-flash"` はそのまま動作します）。

In [ ]:
!pip install -q google-genai

In [ ]:
GC_PROJECT = "your-gcp-project-id"
LOCATION = "global"
MODEL = "gemini-3.8-flash"

from google.colab import auth
from google import genai

auth.authenticate_user()

client = genai.Client(
    vertexai=True,
    project=GC_PROJECT,
    location=LOCATION,
)

response = client.models.generate_content(
    model=MODEL,
    contents="こんにちは！",
)
print(response.text)

### 共有ノートブックでプロジェクト ID のハードコードを避ける方法

ノートブックを Git リポジトリにコミットしたりチームメンバーと共有したりする際は、プロジェクト ID をコードセルに直接記述せず、**Colab シークレット**（左サイドバーの鍵アイコンから `GOOGLE_CLOUD_PROJECT` を登録し「ノートブックからのアクセス」を有効化）から `userdata.get("GOOGLE_CLOUD_PROJECT")` で安全に読み込むパターンが推奨されます。

In [ ]:
LOCATION = "global"
MODEL = "gemini-3.8-flash"

from google.colab import auth, userdata
from google import genai

auth.authenticate_user()

# Colab シークレットから GOOGLE_CLOUD_PROJECT を安全に取得
client = genai.Client(
    vertexai=True,
    project=userdata.get("GOOGLE_CLOUD_PROJECT"),
    location=LOCATION,
)

response = client.models.generate_content(
    model=MODEL,
    contents="宇宙の豆知識を1つ教えてください。",
)
print(response.text)

> **推奨デフォルト `LOCATION = "global"` とリージョンエンドポイントの使い分け:**
> Vertex AI では **`location="global"`** をデフォルトとして推奨します。最新の **Gemini 3.8+** モデル（`gemini-3.8-flash` や `global` 先行で公開される最新 Preview モデルなど）に対応しており、マルチリージョンへの動的なクォータルーティングによって高い可用性を確保できます。一方、データ所在地（データレジデンシー）を日本国内や特定地域に厳格に固定する必要があるワークロードでは、`LOCATION` を `"asia-northeast1"`（東京）や `"us-central1"`（アイオワ）などの**リージョンエンドポイント**に変更してください。

---

## 2. Google AI Studio と Vertex AI の使い分け

`google-genai` SDK は両方のバックエンドを単一のクライアントインターフェースに統合しているため、開発初期は Google AI Studio で素早くプロトタイプを作成し、エンタープライズ向けのセキュリティやガバナンスが必要になった段階で、推論やチャットの呼び出しコードを1行も書き換えることなく Vertex AI へ移行できます。

![図2-2: Google AI Studio と Vertex AI の機能・ガバナンス比較マトリクス](https://examples.drawlib.com/google-genai/images/02_vertex_ai_quickstart/ch02_backend_comparison_ja.png)

*図2-2: Google AI Studio と Vertex AI の機能・ガバナンス比較マトリクス*

### 第3章〜第5章における初期化コードの扱い

- **第3章〜第5章では簡潔な AI Studio 初期化を採用**: 第1章で確認したとおり、Google AI Studio の API キーは Google Cloud プロジェクトに直接リンクして課金やレート制限の引き上げを行えるため、**第3章〜第5章では各サンプルを最小限の見通し良いコードに保つ目的で、簡潔な AI Studio 初期化**（`genai.Client(api_key=userdata.get("GEMINI_API_KEY"))`）に統一しています。
- **Vertex AI ユーザーはそのまま差し替え可能**: 第3章〜第5章のすべてのコードセル（ストリーミング、`ThinkingConfig`、マルチモーダル入力、Pydantic 構造化出力、マルチターンチャット、Function Calling）は、冒頭の `client` 初期化部分を本章の `vertexai=True`（`location="global"`）構成に置き換えるだけで、**残りのコードを1文字も変更せずに Vertex AI 上でそのまま動作します**。